# chap 5

목차
1. XGBoost 기능
2. XGBoost 파라미터 최적화
3. 모델링
4. 힉스 보손 찾기

## XGBoost 기능

1. 누락된 값 처리: missing 매개변수를 통해 자체적으로 누락된 값을 처리할 수 있다. 누락된 값이 들어갈 가능한 노드 분할마다 점수를 매겨서 최대한 높은 점수를 내는 분할을 선택한다.
2. 분할 탐색 알고리즘: 논문 참고 https://arxiv.org/pdf/1603.02754
3. 병렬 컴퓨팅: 데이터를 블록이란 단위로 정렬하므로 여러대의 컴퓨터에 분산 가능
4. 캐시 고려 접근: https://gist.github.com/jboner/2841832 과 논문 참고
5. 블록 압축과 샤딩: 논문 참고
6. 자체적 규제로 정확도 향상

## 2. XGBoost 파라미터 최적화

XGBoost의 전체 목적 함수는 손실 함수와 규제항의 합으로 이루어져 있다
$$\mathcal{L}(\phi) = \sum_{i} l(\hat{y}_i, y_i) + \sum_{k} \Omega(f_k)$$

### 손실함수

### 규제항

## 3. 모델링

### 붓꽃 데이터셋

In [3]:
import numpy as np
import pandas as pd
from sklearn import datasets
iris = datasets.load_iris() # 붓꽃 데이터셋 로드

In [4]:
df = pd.DataFrame(data=np.c_[iris['data'], iris['target']],     # np.c_ # 두 넘파이 배열을 합친다
                  columns=iris['feature_names'] + ['target'])
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0.0
1,4.9,3.0,1.4,0.2,0.0
2,4.7,3.2,1.3,0.2,0.0
3,4.6,3.1,1.5,0.2,0.0
4,5.0,3.6,1.4,0.2,0.0


In [6]:
from sklearn.model_selection import train_test_split
y = df['target']
X = df.drop('target', axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=2
)

In [10]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# 파라미터 세부 내용은 chap6에서
xgb = XGBClassifier(booster='gbtree',              # 기본 학습기
                    objective='multi:softprob', max_depth=6, 
                    learning_rate=0.1, n_estimators=100, n_jobs=-1, eval_metric='mlogloss')

xgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)])
y_pred = xgb.predict(X_test)
score = accuracy_score(y_test, y_pred)
print(f"score: {score}")


[0]	validation_0-mlogloss:0.96661	validation_1-mlogloss:0.97486
[1]	validation_0-mlogloss:0.85681	validation_1-mlogloss:0.86795
[2]	validation_0-mlogloss:0.76341	validation_1-mlogloss:0.77734
[3]	validation_0-mlogloss:0.68313	validation_1-mlogloss:0.69975
[4]	validation_0-mlogloss:0.61356	validation_1-mlogloss:0.63281
[5]	validation_0-mlogloss:0.55288	validation_1-mlogloss:0.57413
[6]	validation_0-mlogloss:0.49966	validation_1-mlogloss:0.52347
[7]	validation_0-mlogloss:0.45281	validation_1-mlogloss:0.47910
[8]	validation_0-mlogloss:0.41140	validation_1-mlogloss:0.44014
[9]	validation_0-mlogloss:0.37468	validation_1-mlogloss:0.40584
[10]	validation_0-mlogloss:0.34216	validation_1-mlogloss:0.37607
[11]	validation_0-mlogloss:0.31315	validation_1-mlogloss:0.34979
[12]	validation_0-mlogloss:0.28722	validation_1-mlogloss:0.32655
[13]	validation_0-mlogloss:0.26401	validation_1-mlogloss:0.30600
[14]	validation_0-mlogloss:0.24320	validation_1-mlogloss:0.28780
[15]	validation_0-mlogloss:0.22450	

### 당뇨병 데이터셋

In [11]:
X, y = datasets.load_diabetes(return_X_y=True)
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor

xgb = XGBRegressor(booster='gbtree', objective='reg:squarederror',
                   max_depth=6, learning_rate=0.1, n_estimators=100, n_jobs=-1)
scores = cross_val_score(xgb, X, y, scoring='neg_root_mean_squared_error', cv=5)
rmse = -scores
print(f"rmse: {np.round(rmse, 3)}")
print(f"rmse 평균: {np.mean(rmse):.3f}")

rmse: [59.397 60.322 69.036 63.211 66.953]
rmse 평균: 63.784


In [12]:
# 비교를 위해 y의 통계량 확인
pd.DataFrame(y).describe()

,0
count,442.000000
mean,152.133484
std,77.093005
min,25.000000
25%,87.000000
50%,140.500000
75%,211.500000
max,346.000000


표준편차 이내이므로 괜찮은 결과임

## 4. 힉스 보손 찾기

In [11]:
from pathlib import Path
data_dir = Path.cwd().parents[2] / "04_Resources" / "Dataset" / "XGBoost"
df = pd.read_csv(data_dir / "atlas-higgs-challenge-2014-v2.csv.gz", nrows=250000, compression='gzip')
df.head()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label,KaggleSet,KaggleWeight
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2.150,0.444,46.062,1.24,-2.475,113.497,0.000814,s,t,0.002653
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,0.725,1.158,-999.000,-999.00,-999.000,46.226,0.681042,b,t,2.233584
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,2.053,-2.028,-999.000,-999.00,-999.000,44.251,0.715742,b,t,2.347389
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,1.660654,b,t,5.446378
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,1.904263,b,t,6.245333


In [12]:
df_d = df.drop(['Weight', 'KaggleSet'], axis=1)
df_d = df_d.rename(columns={"KaggleWeight":"Weight"})
label_col = df_d['Label']
df_d = df_d.drop('Label', axis=1)
df_d['Label'] = label_col

In [13]:
df_d.head()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497,0.002653,s
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226,2.233584,b
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251,2.347389,b
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,5.446378,b
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,6.245333,b


In [14]:
df_d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 33 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   EventId                      250000 non-null  int64  
 1   DER_mass_MMC                 250000 non-null  float64
 2   DER_mass_transverse_met_lep  250000 non-null  float64
 3   DER_mass_vis                 250000 non-null  float64
 4   DER_pt_h                     250000 non-null  float64
 5   DER_deltaeta_jet_jet         250000 non-null  float64
 6   DER_mass_jet_jet             250000 non-null  float64
 7   DER_prodeta_jet_jet          250000 non-null  float64
 8   DER_deltar_tau_lep           250000 non-null  float64
 9   DER_pt_tot                   250000 non-null  float64
 10  DER_sum_pt                   250000 non-null  float64
 11  DER_pt_ratio_lep_tau         250000 non-null  float64
 12  DER_met_phi_centrality       250000 non-null  float64
 13 

In [15]:
df_d['Label'] = df_d['Label'].replace({'s':1, 'b':0})
X = df_d.iloc[:,1:31]
y = df_d.iloc[:,-1]

C:\Users\devsa\AppData\Local\Temp\ipykernel_22956\520882430.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_d['Label'] = df_d['Label'].replace({'s':1, 'b':0})


In [16]:
df_d['test_Weight'] = df_d['Weight'] * 550000 / len(y)

In [17]:
s = np.sum(df_d[df_d['Label']==1]['test_Weight'])
b = np.sum(df_d[df_d['Label']==0]['test_Weight'])
b/s

np.float64(593.9401931492318)

In [19]:
# 1. 모델 객체 생성 (평가 지표 세팅은 여기서 끝냅니다!)
clf = XGBClassifier(
    n_estimators=120, 
    learning_rate=0.1, 
    missing=-999.0,          # 결측치(-999.0) 자동 처리
    scale_pos_weight=b/s,    # 클래스 불균형(Signal vs Background) 해소
    eval_metric=['auc', 'ams@0.15']  # ⭐️ fit 함수에 있던 채점 기준이 이쪽으로 이사 왔습니다!
)

# 2. 모델 학습 (fit에는 순수하게 데이터와 가중치만 들어갑니다)
clf.fit(
    X, y, 
    sample_weight=df_d['test_Weight'],          # 훈련 데이터의 가중치
    eval_set=[(X, y)],                        # 평가용 데이터 세트
    sample_weight_eval_set=[df_d['test_Weight']] # 평가 데이터의 가중치
)
clf.save_model('higgs-sklearn.model')
clf.evals_result()

[0]	validation_0-auc:0.91089	validation_0-ams@0.15:3.81632
[1]	validation_0-auc:0.91497	validation_0-ams@0.15:3.84984
[2]	validation_0-auc:0.91747	validation_0-ams@0.15:4.02133
[3]	validation_0-auc:0.91920	validation_0-ams@0.15:4.14473
[4]	validation_0-auc:0.92039	validation_0-ams@0.15:4.25783
[5]	validation_0-auc:0.92114	validation_0-ams@0.15:4.30765
[6]	validation_0-auc:0.92180	validation_0-ams@0.15:4.35496
[7]	validation_0-auc:0.92228	validation_0-ams@0.15:4.32385
[8]	validation_0-auc:0.92292	validation_0-ams@0.15:4.32770
[9]	validation_0-auc:0.92387	validation_0-ams@0.15:4.35540
[10]	validation_0-auc:0.92450	validation_0-ams@0.15:4.38447
[11]	validation_0-auc:0.92502	validation_0-ams@0.15:4.42677
[12]	validation_0-auc:0.92551	validation_0-ams@0.15:4.44712
[13]	validation_0-auc:0.92603	validation_0-ams@0.15:4.44548
[14]	validation_0-auc:0.92640	validation_0-ams@0.15:4.50608
[15]	validation_0-auc:0.92703	validation_0-ams@0.15:4.54460
[16]	validation_0-auc:0.92737	validation_0-ams@0.1

c:\Users\devsa\anaconda3\Lib\site-packages\xgboost\sklearn.py:1118: UserWarning: [21:42:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)


{'validation_0': OrderedDict([('auc',
               [0.9108909137731261,
                0.9149737185044572,
                0.9174723014301981,
                0.9192029708224471,
                0.9203940085335051,
                0.9211376513413114,
                0.9217967898092925,
                0.9222785063838784,
                0.9229219023580477,
                0.9238680169316417,
                0.9244953049416742,
                0.9250236805694307,
                0.9255127456174714,
                0.9260268167533802,
                0.9264009591431044,
                0.9270345848801917,
                0.9273710462687623,
                0.9278304834943003,
                0.9284345352751293,
                0.9288914864604615,
                0.9294405579334073,
                0.9298507680796411,
                0.9302021632361616,
                0.9305693515828461,
                0.9309325386066278,
                0.9313093849747285,
                0.93160822